<a href="https://colab.research.google.com/github/kurshid1991/Multibiomics/blob/main/1_Biology_and_Data_Foundation/Expression_matrix_dataset_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset Generation:

We simulate RNA-seq counts using negative binomial distribution with biological signal

This dataset mimics real RNA-seq challenges:

- noise
- batch effect
- sparsity
- high dimensionality”

That single framing changes how seriously students take it.

# Whith this dataset

**Learn what is Expression Matrix**

    - Show structure + sparsity

**Preprocess the matrix**

    - Normalize + log transform

**Visualization**

    - PCA separation
    - Heatmaps

**ML Module**
    - Train classifier
    - Show overfitting
**Capstone**
    - Full tumor classification pipeline

In [2]:
import numpy as np
import pandas as pd

# =========================
# REPRODUCIBILITY
# =========================
np.random.seed(42)

# =========================
# PARAMETERS
# =========================
n_genes = 12000
n_samples = 100
n_de_genes = 1000

conditions = np.array(["Tumor"]*50 + ["Normal"]*50)
batches = np.array(["Batch1"]*50 + ["Batch2"]*50)

# =========================
# BASE EXPRESSION
# =========================
base_means = np.random.gamma(shape=2, scale=100, size=n_genes)
dispersion = 0.4

def generate_nb_counts(mean, size):
    p = mean / (mean + size)
    return np.random.negative_binomial(size, 1 - p)

# =========================
# GENERATE MATRIX
# =========================
counts = np.zeros((n_genes, n_samples))
for i in range(n_samples):
    counts[:, i] = generate_nb_counts(base_means, dispersion)

# =========================
# ADD DIFFERENTIAL EXPRESSION
# =========================
de_indices = np.random.choice(n_genes, n_de_genes, replace=False)
upregulated = de_indices[:n_de_genes//2]
downregulated = de_indices[n_de_genes//2:]

counts[upregulated, :50] *= np.random.uniform(2, 6, size=len(upregulated))[:, None]
counts[downregulated, :50] *= np.random.uniform(0.2, 0.5, size=len(downregulated))[:, None]

# =========================
# LIBRARY SIZE EFFECT
# =========================
lib_sizes = np.random.uniform(0.7, 1.5, size=n_samples)
counts = counts * lib_sizes

# =========================
# BATCH EFFECT
# =========================
batch_shift_genes = np.random.choice(n_genes, 800, replace=False)
counts[batch_shift_genes, 50:] *= 1.3

# =========================
# DROPOUT
# =========================
dropout_mask = np.random.rand(n_genes, n_samples) < 0.15
counts[dropout_mask] = 0

# =========================
# FINAL FORMATTING
# =========================
counts = np.round(counts).astype(int)
counts[counts < 0] = 0

gene_ids = [f"Gene_{i}" for i in range(n_genes)]
sample_ids = [f"Sample_{i}" for i in range(n_samples)]

df = pd.DataFrame(counts, index=gene_ids, columns=sample_ids)

meta = pd.DataFrame({
    "sample_id": sample_ids,
    "condition": conditions,
    "batch": batches
})

gene_info = pd.DataFrame({
    "gene_id": gene_ids,
    "type": ["DE" if i in de_indices else "Non-DE" for i in range(n_genes)],
    "regulation": [
        "Up" if i in upregulated else
        "Down" if i in downregulated else
        "None"
        for i in range(n_genes)
    ]
})

df.to_csv("gene_expression_raw_counts.csv")
meta.to_csv("sample_metadata.csv", index=False)
gene_info.to_csv("gene_info.csv", index=False)

print("✅ Python dataset generated successfully!")

✅ Python dataset generated successfully!
